# Sentinel Data Exploration


In [1]:
from pystac import Collection, MediaType
from pystac_client import Client, CollectionClient

import xarray as xr
import dask.array as da
from dask.distributed import Client as DaskClient

In [25]:
eopf_stac_api_root_endpoint = "https://stac.core.eopf.eodc.eu/"
collection = "sentinel-2-l2a"
eopf_catalog = Client.open(url=eopf_stac_api_root_endpoint)

search_result = eopf_catalog.search(
    collections=collection,
    bbox=(11.124756, 47.311058,
          11.459839, 47.463624),
    datetime='2020-05-01T00:00:00Z/2025-05-31T23:59:59.999999Z'
)
id_date_collection = [(item.id, item.datetime) for item in search_result.item_collection()]

id_date_collection

[('S2B_MSIL2A_20250530T101559_N0511_R065_T32TPT_20250530T130924',
  datetime.datetime(2025, 5, 30, 10, 15, 59, 24000, tzinfo=tzutc())),
 ('S2A_MSIL2A_20250527T102041_N0511_R065_T32TPT_20250527T165916',
  datetime.datetime(2025, 5, 27, 10, 20, 41, 24000, tzinfo=tzutc())),
 ('S2B_MSIL2A_20250527T100559_N0511_R022_T32TPT_20250527T155229',
  datetime.datetime(2025, 5, 27, 10, 5, 59, 24000, tzinfo=tzutc())),
 ('S2C_MSIL2A_20250525T101621_N0511_R065_T32TPT_20250525T153015',
  datetime.datetime(2025, 5, 25, 10, 16, 21, 25000, tzinfo=tzutc())),
 ('S2A_MSIL2A_20250524T100701_N0511_R022_T32TPT_20250524T121311',
  datetime.datetime(2025, 5, 24, 10, 7, 1, 24000, tzinfo=tzutc())),
 ('S2C_MSIL2A_20250522T100611_N0511_R022_T32TPT_20250522T153214',
  datetime.datetime(2025, 5, 22, 10, 6, 11, 25000, tzinfo=tzutc())),
 ('S2A_MSIL2A_20250517T101701_N0511_R065_T32TPT_20250517T120915',
  datetime.datetime(2025, 5, 17, 10, 17, 1, 24000, tzinfo=tzutc())),
 ('S2B_MSIL2A_20250517T100559_N0511_R022_T32TPT_20250

In [41]:
c_sentinel2 = eopf_catalog.get_collection(collection)
items = c_sentinel2.get_items(*[item[0] for item in id_date_collection])
assets = [item.get_assets(media_type=MediaType.ZARR) for item in items]
cloud_storage_urls = [asset['product'].href for asset in assets]
cloud_storage_urls

['https://objects.eodc.eu:443/e05ab01a9d56408d82ac32d69a5aae2a:202505-s02msil2a/30/products/cpm_v256/S2B_MSIL2A_20250530T101559_N0511_R065_T32TPT_20250530T130924.zarr',
 'https://objectstore.eodc.eu:2222/e05ab01a9d56408d82ac32d69a5aae2a:202505-s02msil2a/27/products/cpm_v256/S2A_MSIL2A_20250527T102041_N0511_R065_T32TPT_20250527T165916.zarr',
 'https://objectstore.eodc.eu:2222/e05ab01a9d56408d82ac32d69a5aae2a:202505-s02msil2a/27/products/cpm_v256/S2B_MSIL2A_20250527T100559_N0511_R022_T32TPT_20250527T155229.zarr',
 'https://objectstore.eodc.eu:2222/e05ab01a9d56408d82ac32d69a5aae2a:202505-s02msil2a/25/products/cpm_v256/S2C_MSIL2A_20250525T101621_N0511_R065_T32TPT_20250525T153015.zarr',
 'https://objectstore.eodc.eu:2222/e05ab01a9d56408d82ac32d69a5aae2a:202505-s02msil2a/24/products/cpm_v256/S2A_MSIL2A_20250524T100701_N0511_R022_T32TPT_20250524T121311.zarr',
 'https://objectstore.eodc.eu:2222/e05ab01a9d56408d82ac32d69a5aae2a:202505-s02msil2a/22/products/cpm_v256/S2C_MSIL2A_20250522T100611_N0

In [56]:
dt = xr.open_datatree(
    cloud_storage_urls[0],
    engine="zarr",
    chunks="auto"
)
dt

C:\Users\dmitr\AppData\Local\Temp\ipykernel_35456\4010089245.py:1: FutureWarning: In a future version, xarray will not decode the variable 'step' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way of encoding timedelta64 values.
To continue decoding into a timedelta64 dtype, either set `decode_timedelta=True` when opening this dataset, or add the attribute `dtype='timedelta64[ns]'` to this variable on disk.
To opt-in to future behavior, set `decode_timedelta=False`.
  dt = xr.open_datatree(
C:\Users\dmitr\AppData\Local\Temp\ipykernel_35456\4010089245.py:1: FutureWarning: In a future version, xarray will not decode the variable 'step' into a timedelta64 dtype based on the presence of a timedelta-like 'units' attribute by default. Instead it will rely on the presence of a timedelta64 'dtype' attribute, which is now xarray's default way o

<xarray.DataTree>
Group: /
│   Attributes:
│       other_metadata:  {'AOT_retrieval_model': 'SEN2COR_DDV', 'L0_ancillary_dat...
│       stac_discovery:  {'assets': {'analytic': {'eo:bands': [{'center_wavelengt...
├── Group: /conditions
│   ├── Group: /conditions/geometry
│   │       Dimensions:                        (angle: 2, band: 13, y: 23, x: 23,
│   │                                           detector: 6)
│   │       Coordinates:
│   │         * angle                          (angle) <U7 56B 'zenith' 'azimuth'
│   │         * band                           (band) <U3 156B 'b01' 'b02' ... 'b11' 'b12'
│   │         * y                              (y) int64 184B 5300040 5295040 ... 5190040
│   │         * x                              (x) int64 184B 600000 605000 ... 710000
│   │         * detector                       (detector) int64 48B 7 8 9 10 11 12
│   │       Data variables:
│   │           mean_sun_angles                (angle) float64 16B dask.array<chunksize=(2,), meta=np.ndarray>
│   │           mean_viewing_incidence_angles  (band, angle) float64 208B dask.array<chunksize=(13, 2), meta=np.ndarray>
│   │           sun_angles                     (angle, y, x) float64 8kB dask.array<chunksize=(2, 23, 23), meta=np.ndarray>
│   │           viewing_incidence_angles       (band, detector, angle, y, x) float64 660kB dask.array<chunksize=(13, 6, 2, 23, 23), meta=np.ndarray>
│   ├── Group: /conditions/mask
│   │   ├── Group: /conditions/mask/detector_footprint
│   │   │   ├── Group: /conditions/mask/detector_footprint/r10m
│   │   │   │       Dimensions:  (y: 10980, x: 10980)
│   │   │   │       Coordinates:
│   │   │   │         * y        (y) int64 88kB 5300035 5300025 5300015 ... 5190265 5190255 5190245
│   │   │   │         * x        (x) int64 88kB 600005 600015 600025 600035 ... 709775 709785 709795
│   │   │   │       Data variables:
│   │   │   │           b02      (y, x) uint8 121MB dask.array<chunksize=(10980, 10980), meta=np.ndarray>
│   │   │   │           b03      (y, x) uint8 121MB dask.array<chunksize=(10980, 10980), meta=np.ndarray>
│   │   │   │           b04      (y, x) uint8 121MB dask.array<chunksize=(10980, 10980), meta=np.ndarray>
│   │   │   │           b08      (y, x) uint8 121MB dask.array<chunksize=(10980, 10980), meta=np.ndarray>
│   │   │   ├── Group: /conditions/mask/detector_footprint/r20m
│   │   │   │       Dimensions:  (y: 5490, x: 5490)
│   │   │   │       Coordinates:
│   │   │   │         * y        (y) int64 44kB 5300030 5300010 5299990 ... 5190290 5190270 5190250
│   │   │   │         * x        (x) int64 44kB 600010 600030 600050 600070 ... 709750 709770 709790
│   │   │   │       Data variables:
│   │   │   │           b05      (y, x) uint8 30MB dask.array<chunksize=(5490, 5490), meta=np.ndarray>
│   │   │   │           b06      (y, x) uint8 30MB dask.array<chunksize=(5490, 5490), meta=np.ndarray>
│   │   │   │           b07      (y, x) uint8 30MB dask.array<chunksize=(5490, 5490), meta=np.ndarray>
│   │   │   │           b11      (y, x) uint8 30MB dask.array<chunksize=(5490, 5490), meta=np.ndarray>
│   │   │   │           b12      (y, x) uint8 30MB dask.array<chunksize=(5490, 5490), meta=np.ndarray>
│   │   │   │           b8a      (y, x) uint8 30MB dask.array<chunksize=(5490, 5490), meta=np.ndarray>
│   │   │   └── Group: /conditions/mask/detector_footprint/r60m
│   │   │           Dimensions:  (y: 1830, x: 1830)
│   │   │           Coordinates:
│   │   │             * y        (y) int64 15kB 5300010 5299950 5299890 ... 5190390 5190330 5190270
│   │   │             * x        (x) int64 15kB 600030 600090 600150 600210 ... 709650 709710 709770
│   │   │           Data variables:
│   │   │               b01      (y, x) uint8 3MB dask.array<chunksize=(1830, 1830), meta=np.ndarray>
│   │   │               b09      (y, x) uint8 3MB dask.array<chunksize=(1830, 1830), meta=np.ndarray>
│   │   │               b10      (y, x) uint8 3MB dask.array<chunksize=(1830, 1830), meta=np.ndarray>

In [59]:
ds = xr.open_zarr(
    cloud_storage_urls[0],
    chunks={},
    group="/measurements/reflectance/r20m",
    consolidated=True,
)
ds

<xarray.Dataset> Size: 2GB
Dimensions:  (y: 5490, x: 5490)
Coordinates:
  * y        (y) int64 44kB 5300030 5300010 5299990 ... 5190290 5190270 5190250
  * x        (x) int64 44kB 600010 600030 600050 600070 ... 709750 709770 709790
Data variables:
    b01      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b02      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b03      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b04      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b05      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b06      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b07      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b11      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b12      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b8a      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>

In [60]:
ds = ds.assign_coords(time=id_date_collection[0][1])
ds

<xarray.Dataset> Size: 2GB
Dimensions:  (y: 5490, x: 5490)
Coordinates:
  * y        (y) int64 44kB 5300030 5300010 5299990 ... 5190290 5190270 5190250
  * x        (x) int64 44kB 600010 600030 600050 600070 ... 709750 709770 709790
    time     object 8B 2025-05-30T10:15:59.024000+00:00
Data variables:
    b01      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b02      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b03      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b04      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b05      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b06      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b07      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b11      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b12      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>
    b8a      (y, x) float64 241MB dask.array<chunksize=(915, 915), meta=np.ndarray>

In [61]:
ndre = (ds['b8a'] - ds['b05']) / (ds['b8a'] + ds['b05'])
reci = (ds['b07'] / ds['b05']) - 1
msi = ds['b11'] / ds['b8a']

In [34]:
datasets = []
for path in cloud_storage_urls:
    ds = xr.open_zarr(path, chunks={},
                      group="/measurements/reflectance/r20m",
                      consolidated=True)
    t = [next(item[1] for item in id_date_collection if item[0] in path)]
    ds = ds.assign_coords(time=t)
    datasets.append(ds)

combined = xr.concat(datasets, dim="time")

KeyError: '.zmetadata'